# Offline Options Spread Scanner Demo

This notebook walks through the offline options scanner using a previously saved option-chain CSV.

- no live market data is fetched in this notebook
- the scan runs entirely from a local CSV artifact
- the output is for research/demo review only
- this is not financial advice and not a trade recommendation engine

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

TICKER = "SPY"
EXPIRATION = "2026-07-17"
AS_OF = "2026-06-13"
INPUT_CSV = "outputs/yahoo_option_chains/SPY_2026-07-17_yahoo.csv"
OUTPUT_CSV = "outputs/offline_scan_smoke/SPY_2026-07-17_ranked.csv"

REPO_ROOT = Path.cwd()
INPUT_PATH = REPO_ROOT / INPUT_CSV
OUTPUT_PATH = REPO_ROOT / OUTPUT_CSV

print({
    "ticker": TICKER,
    "expiration": EXPIRATION,
    "as_of": AS_OF,
    "input_csv": str(INPUT_PATH),
    "output_csv": str(OUTPUT_PATH),
})

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Expected local input CSV at {INPUT_PATH}. "
        "Generate or copy the saved Yahoo chain CSV before running this demo."
    )

print(f"Found local input CSV: {INPUT_PATH}")

In [ ]:
chain_df = pd.read_csv(INPUT_PATH)

print(f"Input row count: {len(chain_df)}")
print("Columns:", list(chain_df.columns))

display(chain_df.head())

option_type_col = None
for candidate in ("type", "option_type"):
    if candidate in chain_df.columns:
        option_type_col = candidate
        break

if option_type_col is not None:
    display(
        chain_df.groupby(option_type_col)
        .size()
        .rename("row_count")
        .reset_index()
        .sort_values(option_type_col)
    )
else:
    print("No option_type-like column found; skipping input summary by option type.")

## Default Quality Guardrails

The offline scanner applies conservative defaults intended to keep the ranked output useful for research and demo review:

- `max_abs_moneyness_pct = 0.25` filters out legs that are too far from spot
- `max_debit_fraction = 0.67` filters out very high-debit spreads relative to width
- `max_bid_ask_spread_pct = 0.25` filters out legs with very wide quotes
- optional delta, volume, and open-interest filters are only applied when those columns exist in the local file
- if `underlying_price` is missing from the file, the CLI supports `--spot` so the workflow can remain offline

In [ ]:
env = os.environ.copy()
existing_pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(REPO_ROOT) if not existing_pythonpath else os.pathsep.join([str(REPO_ROOT), existing_pythonpath])

command = [
    sys.executable,
    "-m",
    "options_edge_scanner.cli.offline_scan",
    "--input",
    str(INPUT_PATH),
    "--output",
    str(OUTPUT_PATH),
    "--as-of",
    AS_OF,
]

result = subprocess.run(
    command,
    cwd=REPO_ROOT,
    env=env,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.stderr.strip():
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "offline_scan failed. Review stdout/stderr above for details. "
        f"Return code: {result.returncode}"
    )

print(f"Ranked output written to: {OUTPUT_PATH}")

In [ ]:
ranked_df = pd.read_csv(OUTPUT_PATH)

print(f"Output row count: {len(ranked_df)}")
display(ranked_df.head(10))

group_columns = [column for column in ["option_type", "spread_type"] if column in ranked_df.columns]
if group_columns:
    display(
        ranked_df.groupby(group_columns)
        .size()
        .rename("candidate_count")
        .reset_index()
        .sort_values(group_columns)
    )
else:
    print("No option_type/spread_type columns found in ranked output.")

## Simple Visual Checks

These quick charts are meant for workflow inspection, not for trading decisions.

In [ ]:
if "spread_type" in ranked_df.columns and not ranked_df.empty:
    spread_counts = ranked_df["spread_type"].value_counts().sort_index()
    ax = spread_counts.plot(kind="bar", figsize=(7, 4), title="Candidate Count by Spread Type")
    ax.set_xlabel("spread_type")
    ax.set_ylabel("candidate_count")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("No spread_type data available for plotting.")

if "lower_strike" in ranked_df.columns and ranked_df["lower_strike"].notna().any():
    ax = ranked_df["lower_strike"].plot(
        kind="hist",
        bins=min(12, max(4, len(ranked_df))),
        figsize=(7, 4),
        title="Lower Strike Distribution",
    )
    ax.set_xlabel("lower_strike")
    plt.tight_layout()
    plt.show()
elif {"lower_strike", "upper_strike"}.issubset(ranked_df.columns):
    display(ranked_df[["lower_strike", "upper_strike"]].describe())
else:
    print("No strike-range columns available for the second visual check.")

## Interpretation Notes

Guardrails matter because raw option-chain snapshots often contain candidates that are technically rankable but poor for demo review, such as legs far from spot, spreads with very high debit relative to width, or unusually wide quotes.

A single snapshot can still rank heavily toward one side of the market. That does not mean the scanner is recommending those rows, and it does not mean the opposite side was necessarily filtered out incorrectly. Prior validation for this repo showed that both call and put candidates can survive the guardrails when they are near-spot and otherwise valid.

The top-ranked rows should be treated as offline research output to inspect, not as trades to take.

## Demo Takeaway

This project demonstrates a repeatable offline research workflow:

`local chain CSV -> scanner -> guardrails -> ranked review CSV -> inspection`

The notebook is meant to make that workflow easy to explain and reproduce without adding live data dependencies or turning the project into a recommendation engine.